In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-stage-3-2026")

print("Path to dataset files:", path)

In [ ]:
#Part 1 - Task 1: Imports

import os
import torch
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np

import torch.nn.functional as F

In [ ]:
# Part 1 - Task 2: Set dataset paths (based on the folder structure shown)

data_root = os.path.join(path, "PlantVillage")

train_dir = os.path.join(data_root, "train")
test_dir  = os.path.join(data_root, "test")

print("Train dir:", train_dir)
print("Test dir :", test_dir)

print("Train classes folders:", os.listdir(train_dir))
print("Test classes folders :", os.listdir(test_dir))


In [ ]:
#  Part 1 - Task 3: Define transforms
# Requirement: RandomRotation(15) on training + Resize images to 32x32

train_transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
])

test_transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
])


In [ ]:
#   Part 1 - Task 4: Create datasets using ImageFolder

train_dataset = torchvision.datasets.ImageFolder(root=train_dir, transform=train_transform)
test_dataset  = torchvision.datasets.ImageFolder(root=test_dir, transform=test_transform)

print("Classes:", train_dataset.classes)
print("Class to index:", train_dataset.class_to_idx)
print("Train size:", len(train_dataset))
print("Test size:", len(test_dataset))


In [ ]:
#  Part 1 - Task 5: Create DataLoaders

batch_size = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=2
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2
)


In [ ]:
#  Part 1 - Task 6: Display some sample images with labels

def show_batch(loader, class_names, n=8):
    images, labels = next(iter(loader))

    plt.figure(figsize=(10, 4))
    for i in range(n):
        img = images[i].numpy().transpose(1, 2, 0)
        img = np.clip(img, 0, 1)

        plt.subplot(2, 4, i + 1)
        plt.imshow(img)
        plt.title(class_names[labels[i].item()])
        plt.axis("off")

    plt.tight_layout()
    plt.show()

show_batch(train_loader, train_dataset.classes, n=8)


In [ ]:
#Part 2 - Task 1: Define a CNN model class with 5 convolutional layers (+ BatchNorm after each conv)

import torch
import torch.nn as nn
import torch.nn.functional as F

class PotatoCNN(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()

        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)
        self.bn1   = nn.BatchNorm2d(16)

        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.bn2   = nn.BatchNorm2d(32)

        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn3   = nn.BatchNorm2d(64)

        self.conv4 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn4   = nn.BatchNorm2d(128)

        self.conv5 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.bn5   = nn.BatchNorm2d(256)

        self.pool = nn.MaxPool2d(2, 2)

        self.fc1 = nn.Linear(256 * 1 * 1, 128)
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))  # 32x32 -> 16x16
        x = self.pool(F.relu(self.bn2(self.conv2(x))))  # 16x16 -> 8x8
        x = self.pool(F.relu(self.bn3(self.conv3(x))))  # 8x8 -> 4x4
        x = self.pool(F.relu(self.bn4(self.conv4(x))))  # 4x4 -> 2x2
        x = self.pool(F.relu(self.bn5(self.conv5(x))))  # 2x2 -> 1x1

        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x


In [ ]:
# Part 2 - Task 2: Initialize the model and verify output shape

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = PotatoCNN(num_classes=3).to(device)

x, y = next(iter(train_loader))
x = x.to(device)

out = model(x)
print("Batch input shape :", x.shape)
print("Batch output shape:", out.shape)


In [ ]:
# Part 3 - Task 1: Define training loop function

def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in dataloader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    epoch_loss = running_loss / len(dataloader)
    epoch_acc = correct / total

    return epoch_loss, epoch_acc


In [ ]:
# Part 3 - Task 2: Define validation loop function

def validate_one_epoch(model, dataloader, criterion, device):
    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item()

            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    epoch_loss = running_loss / len(dataloader)
    epoch_acc = correct / total

    return epoch_loss, epoch_acc


In [ ]:
# Part 4 - Task 1: Set up device


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

model = model.to(device)


In [ ]:
#   Part 4 - Task 2: Set loss function and optimizer

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)


In [ ]:
# Part 4 - Task 3: Train the model

num_epochs = 5

train_losses = []
val_losses = []
train_accs = []
val_accs = []

for epoch in range(num_epochs):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = validate_one_epoch(model, test_loader, criterion, device)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)

    print(
        f"Epoch [{epoch+1}/{num_epochs}] "
        f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}"
    )


In [ ]:
# Part 4 - Task 4: Plot training and validation losses

plt.figure(figsize=(8, 5))
plt.plot(train_losses, label="Train Loss")
plt.plot(val_losses, label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
#  Part 4 - Task 5: Plot training and validation accuracy

plt.figure(figsize=(8, 5))
plt.plot(train_accs, label="Train Accuracy")
plt.plot(val_accs, label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training vs Validation Accuracy")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
#  Part 5 - Task 1 : CNN with Residual Connection



class PotatoCNN_Residual(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()

        self.conv1 = nn.Conv2d(3, 16, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(16)

        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(32)

        self.conv3 = nn.Conv2d(32, 64, 3, padding=1)
        self.bn3 = nn.BatchNorm2d(64)

        self.conv4 = nn.Conv2d(64, 64, 3, padding=1)
        self.bn4 = nn.BatchNorm2d(64)

        self.conv5 = nn.Conv2d(64, 128, 3, padding=1)
        self.bn5 = nn.BatchNorm2d(128)

        self.pool = nn.MaxPool2d(2, 2)

        #1x1 conv to match channels for residual
        self.skip_conv = nn.Conv2d(32, 64, kernel_size=1)

        self.fc1 = nn.Linear(128 * 1 * 1, 128)
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))   # 32 → 16
        skip = self.pool(F.relu(self.bn2(self.conv2(x))))  # 16 → 8

        x = self.pool(F.relu(self.bn3(self.conv3(skip))))  # 8 → 4
        x = self.pool(F.relu(self.bn4(self.conv4(x))))     # 4 → 2

        #  Fix residual: match channels using 1x1 conv
        skip = self.skip_conv(skip)
        skip = skip[:, :, :2, :2]   # spatial match

        x = x + skip

        x = self.pool(F.relu(self.bn5(self.conv5(x))))     # 2 → 1

        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)

        return x


In [ ]:
model_res = PotatoCNN_Residual(num_classes=3).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model_res.parameters(), lr=1e-3)


In [ ]:
num_epochs = 5

train_losses_res = []
val_losses_res = []
train_accs_res = []
val_accs_res = []

for epoch in range(num_epochs):
    train_loss, train_acc = train_one_epoch(
        model_res, train_loader, criterion, optimizer, device
    )
    val_loss, val_acc = validate_one_epoch(
        model_res, test_loader, criterion, device
    )

    train_losses_res.append(train_loss)
    val_losses_res.append(val_loss)
    train_accs_res.append(train_acc)
    val_accs_res.append(val_acc)

    print(
        f"[Residual] Epoch [{epoch+1}/{num_epochs}] "
        f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}"
    )


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(train_losses_res, label="Train Loss (Residual)")
plt.plot(val_losses_res, label="Val Loss (Residual)")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Residual Model - Training vs Validation Loss")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(train_accs_res, label="Train Acc (Residual)")
plt.plot(val_accs_res, label="Val Acc (Residual)")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Residual Model - Training vs Validation Accuracy")
plt.legend()
plt.grid(True)
plt.show()
